In [ ]:
import pandas as pd
import numpy as np
import os
import pickle

modello = 'o3-mini' # o3-mini   gpt-4o-mini
dataset = 'FOLIO'  # FOLIO    Stanford
mode = 'NL' # NL  FOL

In [ ]:
# MOST SIMILAR

file = open('../Most Similar/Results/Clean Results/' + dataset + '_ms_NL_FOL_' + modello + '_seed.pkl', 'rb')
results = pickle.load(file)
file.close()

file = open('../../perturbations and translations/Most_similar/list_shuffled_positions_ms_' + dataset + '.pkl', 'rb')
perturbs = pickle.load(file)
file.close()

seed_inst_map_answer_ms = {}
counter = 0
for seed in results.keys():
    print(len(results[seed]))
    for instance in results[seed]:
        if instance['NLorFOL'] == mode:
            if dataset == 'FOLIO':            
                counter += 1
                is_correct = instance['answer'] == perturbs[instance['story_id']][instance['number_instance']][0]
                seed_inst_map_answer_ms['s' + str(seed) + 'i' + str(instance['story_id']) + '_' + str(instance['number_instance'])] = is_correct
            elif dataset == 'Stanford': 
                counter += 1
                is_correct = instance['answer'] == perturbs[instance['number_instance']][0]
                seed_inst_map_answer_ms['s' + str(seed) + 'i' + str(instance['number_instance'])] = is_correct

print(len(seed_inst_map_answer_ms), counter)
np.sum(list(seed_inst_map_answer_ms.values())) / len(seed_inst_map_answer_ms)

In [ ]:
# RANKING

file = open('../Ranking/Results/Clean Results/' + dataset + '_r_NL_FOL_' + modello + '_seed.pkl', 'rb')
results = pickle.load(file)
file.close()

file = open('../../perturbations and translations/Ranking/list_shuffled_positions_r_' + dataset + '.pkl', 'rb')
perturbs = pickle.load(file)
file.close()

seed_inst_map_answer_r = {}
counter = 0
for seed in results.keys():
    print(len(results[seed]))
    for instance in results[seed]:
        if instance['NLorFOL'] == mode:
            if dataset == 'FOLIO':      
                counter += 1
                is_correct = sorted(instance['answer'][:2]) == sorted(perturbs[instance['story_id']][instance['number_instance']][:2])
                seed_inst_map_answer_r['s' + str(seed) + 'i' + str(instance['story_id']) + '_' + str(instance['number_instance'])] = is_correct
            elif dataset == 'Stanford': 
                counter += 1
                is_correct = sorted(instance['answer'][:2]) == sorted(perturbs[instance['number_instance']][:2])
                seed_inst_map_answer_r['s' + str(seed) + 'i' + str(instance['number_instance'])] = is_correct
                
print(len(seed_inst_map_answer_r), counter)
np.sum(list(seed_inst_map_answer_r.values())) / len(seed_inst_map_answer_r)

In [ ]:
# Determine degree of alignment between the two tasks

counter_all = 0
counter_ok_ms_ko_r = 0
counter_ko_ms_ok_r = 0
counter_ok_ms_ok_r = 0
counter_ko_ms_ko_r = 0
for case in seed_inst_map_answer_ms.keys():
    if case in seed_inst_map_answer_r.keys():
        counter_all += 1
        if seed_inst_map_answer_ms[case] and (not seed_inst_map_answer_r[case]):
            counter_ok_ms_ko_r += 1
        if (not seed_inst_map_answer_ms[case]) and seed_inst_map_answer_r[case]:
            counter_ko_ms_ok_r += 1
        if seed_inst_map_answer_ms[case] and seed_inst_map_answer_r[case]:
            counter_ok_ms_ok_r += 1
        if (not seed_inst_map_answer_ms[case]) and (not seed_inst_map_answer_r[case]):
            counter_ko_ms_ko_r += 1

print(counter_all)
print("ok_ms_ko_r", counter_ok_ms_ko_r/counter_all)
print("ko_ms_ok_r", counter_ko_ms_ok_r/counter_all)
print("ok_ms_ok_r", counter_ok_ms_ok_r/counter_all)
print("ko_ms_ko_r", counter_ko_ms_ko_r/counter_all)